# Lesson 4.2 — Subcharts & Global Values

**Phase 4: Dependencies & Hooks**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Understand how values are scoped in parent/child chart relationships
- Use `global:` values to share configuration across all subcharts
- Author your own subchart and bundle it under `charts/`
- Use `import-values` to import child chart values into the parent

---

## Parent/Child Value Scoping

When a chart has sub-charts, value scoping works like this:

```
Parent values.yaml:
  myvalue: "parent"   → visible only to parent
  child1:             → subchart named 'child1' sees these as its .Values.*
    port: 8080
  global:             → ALL charts (parent + all subcharts) see these
    imageRegistry: "registry.example.com"
```

**Key rules:**
- Parent can read child values via `.Values.child1.port`
- Child **cannot** read parent's values (only its own namespace + global)
- `global:` flows downward to all subcharts

> **Reference:** [Subcharts and Global Values](https://helm.sh/docs/chart_template_guide/subcharts_and_globals/) — https://helm.sh/docs/chart_template_guide/subcharts_and_globals/

---

## Building a Subchart

Let's create a parent chart with a hand-written subchart to see the scoping in action:

In [ ]:
# Create a parent chart
rm -rf /tmp/parentchart
helm create /tmp/parentchart
rm -rf /tmp/parentchart/templates/*

# Create a subchart manually inside charts/
mkdir -p /tmp/parentchart/charts/mycache/templates

cat > /tmp/parentchart/charts/mycache/Chart.yaml << 'EOF'
apiVersion: v2
name: mycache
version: 0.1.0
appVersion: "1.0"
EOF

cat > /tmp/parentchart/charts/mycache/values.yaml << 'EOF'
port: 6379
maxMemory: "128mb"
EOF

cat > /tmp/parentchart/charts/mycache/templates/configmap.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-mycache-config
data:
  # .Values here is SCOPED to mycache — only sees mycache's values
  port: {{ .Values.port | quote }}
  maxMemory: {{ .Values.maxMemory | quote }}

  # .Values.global is available from the parent's global: section
  imageRegistry: {{ .Values.global.imageRegistry | default "docker.io" | quote }}
EOF

echo "Subchart created:"
find /tmp/parentchart/charts/mycache -type f

In [ ]:
# Parent values.yaml
cat > /tmp/parentchart/values.yaml << 'EOF'
# Parent-only values
appName: myapp

# Global values — flow DOWN to all subcharts
global:
  imageRegistry: "my-registry.company.com"

# Values scoped to the 'mycache' subchart
# These OVERRIDE mycache's own values.yaml defaults
mycache:
  port: 6380          # Override default 6379
  maxMemory: "256mb"  # Override default 128mb
EOF

# Parent template
cat > /tmp/parentchart/templates/configmap.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-parent-config
data:
  # Parent reads its own values
  appName: {{ .Values.appName | quote }}
  # Parent can also read child values
  cachePort: {{ .Values.mycache.port | quote }}
  # Global is accessible here too
  imageRegistry: {{ .Values.global.imageRegistry | quote }}
EOF

helm template my-release /tmp/parentchart

---

## Global Values in Action

Notice that in the output:
- The parent's ConfigMap shows `imageRegistry: my-registry.company.com`
- The subchart's ConfigMap **also** shows `imageRegistry: my-registry.company.com` — because `global:` propagates down

This is the key benefit of `global:` — one place to configure things that all charts need.

In [ ]:
# Override the global imageRegistry at install time
helm template my-release /tmp/parentchart \
  --set global.imageRegistry="staging-registry.company.com"

echo ""
echo "Both parent and child should show the new registry"

---

## `import-values` — Promote Child Values to Parent

Sometimes a subchart computes or provides values that the parent needs. Use `import-values` to pull them up:

In [ ]:
# Demonstrate the exports/import-values pattern
cat << 'EOF'
# Child chart's values.yaml with an 'exports' key
exports:
  serviceConfig:
    host: mycache-service
    port: 6379

# ---

# Parent Chart.yaml — import the child's exports
dependencies:
  - name: mycache
    version: "0.1.0"
    repository: "file://charts/mycache"
    import-values:
      - serviceConfig      # imports exports.serviceConfig into parent's .Values.serviceConfig

# Result: parent templates can use .Values.serviceConfig.host and .Values.serviceConfig.port
EOF

---

## Common Patterns with Subcharts

### Pattern 1: Registry prefix in global
```yaml
global:
  imageRegistry: "internal-registry.company.com"
```
All subcharts prefix their image pulls with this registry — one change = all charts updated.

### Pattern 2: StorageClass in global
```yaml
global:
  storageClass: "fast-ssd"
```
All PVCs across all subcharts use the same storage class.

### Pattern 3: Conditional deployment of subcharts
```yaml
postgresql:
  enabled: true   # In prod: use the bundled postgresql
# For production with RDS: set postgresql.enabled=false
# and point your app at the external DB via env vars
```

---

## Exercise

In [ ]:
# TASK 1: Add a second subchart 'myworker' under /tmp/parentchart/charts/
#   that has its own values (workerCount: 2, queue: default).
#   Make it use global.imageRegistry in a ConfigMap template.
# TODO:


# TASK 2: In the parent's values.yaml, override myworker.workerCount to 5.
#   Render the chart and verify the parent-supplied value is used.
# TODO:


# TASK 3: Add a 'global.environment' value and use it in both the parent
#   and mycache templates. Show that changing it with --set affects all resources.
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

- Parent chart values are scoped: `subchart-name:` block → subchart only.
- `global:` block flows down to all subcharts.
- Subcharts cannot read parent values — only their own namespace + global.
- `import-values` allows a child's exported values to be promoted to the parent scope.
- Common global values: `imageRegistry`, `storageClass`, `imagePullSecrets`.

---

## Next Lesson

➡️ [Lesson 4.3 — Hooks & Release Lifecycle](lesson-4.3-hooks.ipynb)